# Feature Engineering for Sports vs Politics Classification

This notebook focuses on text feature extraction using various techniques: Bag of Words, TF-IDF, and N-grams, followed by data splitting for model training.

## Section 1: Import Libraries and Load Data

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from preprocess import clean_text, preprocess_batch

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

In [ ]:
# Load processed data
df = pd.read_csv('../data/processed/sports_politics.csv')

print(f"Dataset loaded: {df.shape}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\nFirst sample text:")
print(df['text'].iloc[0])

## Section 2: Text Preprocessing and Cleaning

In [ ]:
# Show before and after preprocessing
print("TEXT PREPROCESSING EXAMPLES")
print("=" * 100)

# Sample some texts from each category
sports_sample = df[df['label'] == 0]['text'].iloc[0]
politics_sample = df[df['label'] == 1]['text'].iloc[0]

samples = [
    ("SPORTS", sports_sample),
    ("POLITICS", politics_sample)
]

for category, text in samples:
    cleaned = clean_text(text)
    print(f"\n{category}:")
    print(f"Original: {text[:150]}...")
    print(f"Cleaned:  {cleaned[:150]}...")
    print("-" * 100)

# Apply preprocessing to all texts
print("\nApplying preprocessing to all texts...")
df['cleaned_text'] = df['text'].apply(clean_text)
print("✓ Preprocessing complete!")

## Section 3: Bag of Words (CountVectorizer)

In [ ]:
# Bag of Words using CountVectorizer
print("Creating Bag of Words features...")
bow_vectorizer = CountVectorizer(max_features=5000, min_df=5, max_df=0.8)
X_bow = bow_vectorizer.fit_transform(df['cleaned_text'])

print(f"✓ Bag of Words shape: {X_bow.shape}")
print(f"  - Samples: {X_bow.shape[0]}")
print(f"  - Features (vocabulary size): {X_bow.shape[1]}")

# Get feature names
bow_features = bow_vectorizer.get_feature_names_out()
print(f"\nSample features (first 20 words): {list(bow_features[:20])}")

## Section 4: TF-IDF Features

In [ ]:
# TF-IDF Features
print("\nCreating TF-IDF features...")
tfidf_vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8)
X_tfidf = tfidf_vectorizer.fit_transform(df['cleaned_text'])

print(f"✓ TF-IDF shape: {X_tfidf.shape}")
print(f"  - Samples: {X_tfidf.shape[0]}")
print(f"  - Features: {X_tfidf.shape[1]}")

# Get top TF-IDF features for each class
tfidf_features = tfidf_vectorizer.get_feature_names_out()
tfidf_matrix = X_tfidf.toarray()

# Average TF-IDF for SPORTS and POLITICS
sports_tfidf = tfidf_matrix[df['label'] == 0].mean(axis=0)
politics_tfidf = tfidf_matrix[df['label'] == 1].mean(axis=0)

# Top 10 features for each class
top_sports_idx = np.argsort(sports_tfidf)[-10:][::-1]
top_politics_idx = np.argsort(politics_tfidf)[-10:][::-1]

print(f"\nTop 10 TF-IDF features for SPORTS:")
print([tfidf_features[i] for i in top_sports_idx])

print(f"\nTop 10 TF-IDF features for POLITICS:")
print([tfidf_features[i] for i in top_politics_idx])

## Section 5: N-gram Features

In [ ]:
# N-grams (Unigrams and Bigrams)
print("\nCreating N-gram features...")

# Unigrams (1-grams) - Already done with TF-IDF
print(f"✓ Unigrams (TF-IDF): {X_tfidf.shape}")

# Bigrams
tfidf_bigrams = TfidfVectorizer(ngram_range=(2, 2), max_features=5000, min_df=5, max_df=0.8)
X_bigrams = tfidf_bigrams.fit_transform(df['cleaned_text'])
print(f"✓ Bigrams (TF-IDF): {X_bigrams.shape}")

# Combined uni+bigrams
tfidf_combined = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, min_df=5, max_df=0.8)
X_combined = tfidf_combined.fit_transform(df['cleaned_text'])
print(f"✓ Combined (1-2 grams TF-IDF): {X_combined.shape}")

# Display sample bigrams
bigram_features = tfidf_bigrams.get_feature_names_out()
print(f"\nSample bigrams (first 15): {list(bigram_features[:15])}")

## Section 6: Compare Feature Dimensions

In [ ]:
# Comparison of different feature representations
feature_comparison = pd.DataFrame({
    'Feature Type': ['Bag of Words', 'TF-IDF (unigrams)', 'TF-IDF (bigrams)', 'TF-IDF (1-2 grams)'],
    'Samples': [X_bow.shape[0], X_tfidf.shape[0], X_bigrams.shape[0], X_combined.shape[0]],
    'Features': [X_bow.shape[1], X_tfidf.shape[1], X_bigrams.shape[1], X_combined.shape[1]],
    'Sparsity (%)': [
        round(100 * (1 - X_bow.nnz / (X_bow.shape[0] * X_bow.shape[1])), 2),
        round(100 * (1 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])), 2),
        round(100 * (1 - X_bigrams.nnz / (X_bigrams.shape[0] * X_bigrams.shape[1])), 2),
        round(100 * (1 - X_combined.nnz / (X_combined.shape[0] * X_combined.shape[1])), 2)
    ]
})

print("\nFEATURE REPRESENTATION COMPARISON")
print("=" * 80)
print(feature_comparison.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Features comparison
axes[0].bar(feature_comparison['Feature Type'], feature_comparison['Features'], color='skyblue')
axes[0].set_title('Number of Features by Type', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Features')
axes[0].tick_params(axis='x', rotation=45)

# Sparsity comparison
axes[1].bar(feature_comparison['Feature Type'], feature_comparison['Sparsity (%)'], color='lightcoral')
axes[1].set_title('Sparsity by Feature Type', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Sparsity (%)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../results/feature_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Feature comparison chart saved!")

## Section 7: Train-Test Split

In [ ]:
# Split data into train and test sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,  # Using TF-IDF features for modeling
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print("Train-Test Split (80-20)")
print("=" * 50)
print(f"Training set size: {X_train.shape}")
print(f"  - Samples: {X_train.shape[0]}")
print(f"  - Features: {X_train.shape[1]}")

print(f"\nTest set size: {X_test.shape}")
print(f"  - Samples: {X_test.shape[0]}")
print(f"  - Features: {X_test.shape[1]}")

print(f"\nLabel distribution in training set:")
print(y_train.value_counts().sort_index())

print(f"\nLabel distribution in test set:")
print(y_test.value_counts().sort_index())

# Visualization of train-test split
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Training set distribution
y_train.value_counts().sort_index().plot(kind='bar', ax=axes[0], color=['#FF6B6B', '#4ECDC4'])
axes[0].set_title('Training Set Label Distribution', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['SPORTS', 'POLITICS'], rotation=0)
axes[0].set_ylabel('Count')

# Test set distribution
y_test.value_counts().sort_index().plot(kind='bar', ax=axes[1], color=['#FF6B6B', '#4ECDC4'])
axes[1].set_title('Test Set Label Distribution', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(['SPORTS', 'POLITICS'], rotation=0)
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../results/train_test_split.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Train-test split visualization saved!")

## Section 8: Save Processed Features

In [ ]:
import pickle
from pathlib import Path

# Create output directory
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

# Save train-test split data
with open(output_dir / 'X_train.pkl', 'wb') as f:
    pickle.dump(X_train, f)
with open(output_dir / 'X_test.pkl', 'wb') as f:
    pickle.dump(X_test, f)
with open(output_dir / 'y_train.pkl', 'wb') as f:
    pickle.dump(y_train, f)
with open(output_dir / 'y_test.pkl', 'wb') as f:
    pickle.dump(y_test, f)

# Save vectorizers
with open(output_dir / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

print("✓ Feature engineering complete!")
print(f"✓ Saved train-test split files to {output_dir}")
print("\nFiles saved:")
print("  - X_train.pkl, X_test.pkl")
print("  - y_train.pkl, y_test.pkl")
print("  - tfidf_vectorizer.pkl")